# Portfolio equality workshop

Retain 4 layers / return scale 1.5 / no penalty / 30 calls; 3 layers / scale 1.7 / no penalty / 10 calls; 3 layers / scale 10 / penalty 2 / 40 calls. All three assets are unsigned 2-bit values. The first two scenarios are unconstrained despite the declared budget 6. The last has a soft equality penalty. Its draft cell spells PENALTY differently from the used Penalty; the native model uses the final complete solution value 2.

Preserve every original scenario and problem size. Users supply high-level
costs and classical parameters; the native qfunc prepares uniform registers,
applies positive cost phases, and mixes with RX(beta). No physical wires or bit
decoding are required in the workflow. The companion portfolio_programs.py shows
the complete native quantum functions.

The source uses SciPy COBYLA, whose maxiter limits objective evaluations. Native
COBYLA counts trust-region iterations, so this workflow stops BEFORE an extra
objective execution at the source cap, selects the best actually evaluated
parameters, and reports evaluation_budget and nonconvergence. It does not invent
an optimizer return value or claim the same optimization trajectory. Exact local
mean probabilities replace provider shot-noisy estimates; final 1,000 seeded
samples are actual draws. Sampling/readout and independent verification are
outside the objective-call cap. Classical exhaustive minima are comparison labels,
never substituted for quantum observations. Penalties stay soft; results retain
infeasible assignments. No remote service, hardware or advantage claim.


## Portfolio Optimization with the Quantum Approximate Optimization Algorithm (QAOA)

Native counterpart of original code cell 4.


In [ ]:
from io import BytesIO

from IPython.display import Image, display
from matplotlib import pyplot as plt
from matplotlib.figure import Figure

from examples.synthesis.portfolio_reference import COVARIANCE, RETURNS
from examples.synthesis.tutorials.workshops.finance_workshops.portfolio_programs import (
    holdings_program,
)
from examples.synthesis.tutorials.workshops.finance_workshops.portfolio_workflow import (
    EQUALITY_SCENARIOS,
    assignments,
    bindings,
    build_program,
    classical_optima,
    cost,
    distribution,
    expected_return,
    initial_parameters,
    mean_cost,
    plot_samples,
    plot_trace,
    risk,
    run_workflow,
    verify,
)
from flagquantum.drawer import draw


def show_figure(figure: Figure) -> None:
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=130)
    display(Image(buffer.getvalue()))
    plt.close(figure)


## The Portfolio Optimization Problem Parameters

Native counterpart of original code cell 7.


In [ ]:
print("Expected returns:", RETURNS)
print("Covariance:", COVARIANCE)
total_budget = 6


## Defining the Variables

Native counterpart of original code cell 9.


In [ ]:
scenario = EQUALITY_SCENARIOS[0]
print("Three two-bit assets, domain size:", len(assignments(scenario)))


## Define the Expected Return

Native counterpart of original code cell 11.


In [ ]:
print("Example expected return:", expected_return((1, 2, 1)))


## Define the Risk Term

Native counterpart of original code cell 13.


In [ ]:
print("Example risk:", risk((1, 2, 1)))


## Define the Entire Portfolio Optimization Objective Function

Native counterpart of original code cell 15.


In [ ]:
print(scenario)
print("Unconstrained objective:", cost(scenario, (1, 2, 1)))
assert scenario.penalty == 0


## Build the QAOA Circuit

Native counterpart of original code cell 17.


In [ ]:
print("Native qfunc:", holdings_program)


## Build the QAOA Circuit

Native counterpart of original code cell 18.


In [ ]:
initial = initial_parameters(scenario)
program = build_program(scenario, initial)


## Synthesizing and Visualizing

Native counterpart of original code cell 20.


In [ ]:
print(program.report["resources"])
print(draw(program.circuit))


## Execution and Post Processing

Native counterpart of original code cell 22.


In [ ]:
shots = 1000
print("Interleaved gamma and beta:", initial)


## Define a Callback Function to Track the Optimization

Native counterpart of original code cell 24.


In [ ]:
print("Record every executed objective, not native trust-region iterations.")
print("Source objective-call cap:", scenario.evaluation_budget)


## Define the Objective Function

Native counterpart of original code cell 26.


In [ ]:
probabilities = distribution(scenario, program)
print("Exact local mean estimate:", mean_cost(scenario, probabilities))


## Optimize

Native counterpart of original code cell 28.


In [ ]:
intro = run_workflow(scenario, shots=shots, seed=1940)
print("Termination:", intro.termination, "Converged:", intro.converged)
print("Actual objective executions:", len(intro.trace))


## Look at the Results

Native counterpart of original code cell 30.


In [ ]:
for row in intro.samples:
    print(row)
print("Classical objective minima:", classical_optima(scenario))


## Convergence Graph

Native counterpart of original code cell 32.


In [ ]:
show_figure(plot_trace(intro))
show_figure(plot_samples(intro))


## The Entire Solution

Native counterpart of original code cell 34.


In [ ]:
unconstrained = run_workflow(EQUALITY_SCENARIOS[1], seed=1941)
print(unconstrained.scenario)
print(unconstrained.program.report["resources"])
print(unconstrained.termination, unconstrained.converged)
print("Actual objective evaluations:", len(unconstrained.trace))
for row in unconstrained.samples:
    print(row)
print(
    "Independent classical objective minima:", classical_optima(unconstrained.scenario)
)
show_figure(plot_trace(unconstrained))
show_figure(plot_samples(unconstrained))


## Define the Objective Value with a Penalty Term

Native counterpart of original code cell 38.


In [ ]:
constrained_scenario = EQUALITY_SCENARIOS[2]
print("Soft equality penalty:", constrained_scenario.penalty)
print("Budget:", total_budget)
print("Cost for holdings outside the budget:", cost(constrained_scenario, (3, 3, 3)))


## Solution

Native counterpart of original code cell 41.


In [ ]:
equality = run_workflow(EQUALITY_SCENARIOS[2], seed=1942)
print(equality.scenario)
print(equality.program.report["resources"])
print(equality.termination, equality.converged)
print("Actual objective evaluations:", len(equality.trace))
for row in equality.samples:
    print(row)
print("Independent classical objective minima:", classical_optima(equality.scenario))
show_figure(plot_trace(equality))
show_figure(plot_samples(equality))
workflows = (
    intro,
    unconstrained,
    equality,
)
checks = [
    verify(w.scenario, parameters)
    for w in workflows
    for parameters in (*bindings(w.scenario), w.selected.parameters)
]
assert all(check.maximum_state_error < 3e-11 for check in checks)
assert all(check.dirty_workspace_probability < 1e-22 for check in checks)
for workflow in workflows:
    print(workflow.scenario.name, workflow.termination, workflow.converged)
    print("Actual objective evaluations:", len(workflow.trace))
    print("Initial and selected mean:", workflow.trace[0].cost, workflow.selected.cost)
    print(
        "Best observed (not overwritten by classical optimum):", workflow.best_observed
    )
    print("Independent classical minima:", classical_optima(workflow.scenario))
    assert len(workflow.trace) <= workflow.scenario.evaluation_budget
    assert sum(row.count for row in workflow.samples) == 1000
